# Alberi di regressione e random forest

*Metodi Matematici e Statistici per le Decisioni Economiche · A.A. 2026/27 · Lezione 4*

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincnardelli/msd/blob/main/codice/4/alberi_regressione.ipynb)

Il codice della lezione, sul dataset `cigarette.csv`. Gli alberi e le random forest li stimiamo con
**scikit-learn**: vuole le variabili esplicative in una tabella `X` e la risposta in `y`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

DATI = "https://www.vincnardelli.com/msd/dati/"

In [ ]:
cig = pd.read_csv(DATI + "cigarette.csv")
y = cig["packpc"]

## Un albero con un solo predittore

In [ ]:
X = cig[["taxs"]]
reg_tree = DecisionTreeRegressor(max_depth=2, random_state=1).fit(X, y)

plt.figure(figsize=(10, 5))
plot_tree(reg_tree, feature_names=X.columns, filled=True, rounded=True)
plt.show()

In [ ]:
x = pd.DataFrame({"taxs": np.linspace(cig["taxs"].min(), cig["taxs"].max(), 500)})
plt.scatter(cig["taxs"], y, s=8)
plt.plot(x["taxs"], reg_tree.predict(x), color="red")   # previsione a gradini
plt.xlabel("taxs")
plt.ylabel("packpc")
plt.show()

## Più predittori: `taxs`, `income`, `pop`

`min_samples_leaf=5` impone almeno 5 osservazioni per foglia, `max_depth` limita la profondità.

In [ ]:
X = cig[["taxs", "income", "pop"]]
reg_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=1).fit(X, y)

plt.figure(figsize=(16, 7))
plot_tree(reg_tree, feature_names=X.columns, filled=True, rounded=True, fontsize=8)
plt.show()

## Quanto deve essere profondo l'albero? Validazione incrociata

In [ ]:
from sklearn.model_selection import cross_val_score

profondita = range(1, 11)
mse_cv = [-cross_val_score(DecisionTreeRegressor(max_depth=d, min_samples_leaf=5, random_state=1),
                           X, y, cv=10, scoring="neg_mean_squared_error").mean() for d in profondita]
plt.plot(profondita, mse_cv, marker="o")
plt.xlabel("profondità massima")
plt.ylabel("MSE (validazione incrociata)")
plt.show()

## Confronto con il modello lineare

In [ ]:
lm_model = smf.ols("packpc ~ taxs + income + pop", data=cig).fit()
print(lm_model.summary())

lm_mse = np.mean((y - lm_model.predict(cig)) ** 2)
tree_mse = np.mean((y - reg_tree.predict(X)) ** 2)
print("MSE lineare:", lm_mse)
print("MSE albero: ", tree_mse)

## Random forest

L'MSE della random forest lo calcoliamo sulle previsioni **out-of-bag**: ogni osservazione è prevista solo dagli
alberi che non l'hanno vista. Per questo è più onesto dell'MSE sui dati di training dei due modelli sopra.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=500, oob_score=True, random_state=123).fit(X, y)
rf_mse = np.mean((y - rf_model.oob_prediction_) ** 2)
print("MSE random forest (OOB):", rf_mse)

In [ ]:
pd.Series(rf_model.feature_importances_, index=X.columns).sort_values().plot.barh()
plt.xlabel("importanza (riduzione media dell'errore)")
plt.show()